# 05 — The Probability Simplex

See the space of *all* categorical distributions as one geometric object — a triangle for three outcomes — and learn its anatomy by hand: where the certain outcomes sit, where the uniform sits, what an edge means, why a mixture never leaves it, how a distribution *is* its own barycentric coordinates. Then draw on it the quantities of `02/01`–`02/03` — entropy as a hill, relative entropy as an asymmetric bowl, the independence surface that mutual information measures the distance to — and finally ask the question that opens the geometry arc: **how far apart are two points of this triangle?** You already own four candidate answers, and they disagree. Deciding between them is `02/06`–`02/08`.

**Prerequisites:** `02/01_shannon_entropy` (entropy, concavity, the Jensen check on random points "of the simplex", `dirichlet(1,…,1)`), `02/02_kl_divergence` (the two landscapes of $D$ on the triangle, convexity), `02/03_mutual_information` (product of marginals, $I = D(p_{XY}\|p_X\otimes p_Y)$), `01/14_trace_distance_and_fidelity` (total variation and the Bhattacharyya overlap on probability vectors), `01/12`–`01/13` (the Bloch ball: density matrices as a convex body, the diagonal slice).

**What you'll be able to do**
- Define the simplex $\Delta_{n-1}$ by its two constraints, compute its dimension, and prove it is convex — a mixture of distributions is a distribution — so the concavity of `02/01` and the convexity of `02/02` are statements *about a convex set*.
- Map a three-outcome distribution to a point of the triangle by hand (barycentric coordinates: the weights *are* the probabilities), read the anatomy — vertices, edges, centre — and see that the uniform is the centre of mass and `dirichlet(1,1,1)` is the uniform law on the triangle.
- Draw entropy as a concave hill and relative entropy as an asymmetric bowl on the triangle, and place the two-bit tables of `01/16` in the tetrahedron of joint tables, where the product tables form a curved *independence surface*.
- Draw the level sets of four candidate distances around one point — Euclidean, total variation, Hellinger, KL — and see them agree at the centre and disagree near the boundary, which is the question the rest of the arc answers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence, mutual_information, shannon_entropy

rng = np.random.default_rng(0)   # random points of the simplex (section 2) and random mixtures (section 1)
viz.use_course_style()

## 1. The set, its dimension, and why it is convex

A distribution on $n$ outcomes is a vector $p = (p_1, \dots, p_n)$ with two constraints: $p_i \ge 0$ for every $i$, and $\sum_i p_i = 1$. The set of all of them is the **probability simplex**

$$
\Delta_{n-1} = \Big\{\,p \in \mathbb R^n : p_i \ge 0,\ \textstyle\sum_i p_i = 1\,\Big\} .
$$

Count the freedom: $n$ numbers, one linear equation, so $n - 1$ dimensions — the subscript. For $n = 2$ a segment (the coins of `01/01`, $p = (p, 1-p)$, one number); for $n = 3$ a **triangle**; for $n = 4$ a tetrahedron; in general a polytope with $n$ **vertices** $\delta_1 = (1, 0, \dots, 0), \dots, \delta_n$ — the certain outcomes.

**Convex, by hand.** Take two points $p, q \in \Delta_{n-1}$ and $\lambda \in [0, 1]$. The mixture $r = \lambda p + (1-\lambda) q$ has $r_i = \lambda p_i + (1-\lambda)q_i \ge 0$ (a sum of non-negative terms) and $\sum_i r_i = \lambda\cdot 1 + (1-\lambda)\cdot 1 = 1$. So $r \in \Delta_{n-1}$: *the segment between two distributions is made of distributions* — the simplex is a **convex set**, and "mixing" is moving along a straight line inside it. That is the set on which `02/01` proved entropy concave (a hill with no dips along any segment) and `02/02` proved relative entropy jointly convex.

**Barycentric coordinates.** Every $p$ is a mixture of the vertices — $p = \sum_i p_i\,\delta_i$, with *its own entries as the weights*, since $\sum_i p_i = 1$. So to draw the triangle, choose three points $v_1, v_2, v_3$ in the plane for the vertices and send $p \mapsto \sum_i p_i v_i$: the probabilities become the **barycentric coordinates** of the point. Put a mass $p_i$ at each corner and the point is their centre of mass. The course helper `viz.plot_simplex_points` does exactly that with an equilateral triangle; let's do it by hand first, check the helper agrees, and verify convexity on random mixtures.


In [ ]:
# the three vertices of an equilateral triangle in the plane, and the barycentric map p -> sum_i p_i v_i
V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])
def to_plane(p):
    p = np.asarray(p, dtype=float)
    return p @ V                                   # centre of mass of the vertices with weights p

examples = {
    "uniform (1/3, 1/3, 1/3)": np.array([1/3, 1/3, 1/3]),
    "p = (0.7, 0.2, 0.1)":      np.array([0.7, 0.2, 0.1]),
    "almost certain (0.92, 0.04, 0.04)": np.array([0.92, 0.04, 0.04]),
    "on an edge (0.5, 0.5, 0)": np.array([0.5, 0.5, 0.0]),
}
for name, p in examples.items():
    print(f"{name:36s} -> plane point {np.round(to_plane(p), 4)}   (entries >= 0: {np.all(p >= 0)}, sum = {p.sum():.3f})")
print("the centre of mass of the three vertices:", np.round(V.mean(axis=0), 4), "= the uniform's point")
def draw_triangle(ax, labels=True):
    # the empty triangle, by hand: three edges and three labelled vertices
    tri = np.vstack([V, V[:1]])
    ax.plot(tri[:, 0], tri[:, 1], color=COLORS["muted"], lw=1.5, zorder=1)
    if labels:
        for (x, y), lab, dx, dy in zip(V, [r"$\delta_1=(1,0,0)$", r"$\delta_2=(0,1,0)$", r"$\delta_3=(0,0,1)$"], [-0.07, 0.07, 0.0], [-0.045, -0.045, 0.045]):
            ax.text(x + dx, y + dy, lab, ha="center", va="center", fontsize=11, color=COLORS["text"])
    ax.set_aspect("equal"); ax.axis("off")

# convexity: 2000 random mixtures of random pairs never leave the simplex
p_r, q_r = rng.dirichlet(np.ones(3), size=2000), rng.dirichlet(np.ones(3), size=2000)
lam = rng.uniform(size=(2000, 1))
mix = lam * p_r + (1 - lam) * q_r
print(f"2000 random mixtures: min entry = {mix.min():.4f} (>= 0), max |sum - 1| = {np.max(np.abs(mix.sum(1) - 1)):.1e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6.2), gridspec_kw={"wspace": 0.15})
draw_triangle(axes[0])
for k, (name, p) in enumerate(examples.items()):
    axes[0].scatter(*to_plane(p), s=120, color=[viz.SOURCE_COLOR, viz.TARGET_COLOR, viz.FLOW_COLOR, COLORS["quantum"]][k], edgecolor="white", zorder=3)
    axes[0].annotate(name, to_plane(p), xytext=(9, 6), textcoords="offset points", fontsize=10)
axes[0].set_title("By hand: the barycentric map p -> sum_i p_i v_i", pad=26)
viz.plot_simplex_points(examples, ax=axes[1], title="The course helper, same triangle")
plt.show()

**Read the output and the figure.** The by-hand barycentric map puts the uniform at the centre of mass of the three corners, and the course helper (right) draws the same triangle with the same points — every example lands where its weights say: $(0.7, 0.2, 0.1)$ pulled toward $\delta_1$, the almost-certain one hugging that corner, $(0.5, 0.5, 0)$ exactly on the bottom edge, halfway between $\delta_1$ and $\delta_2$. Two thousand random mixtures stay inside to machine precision: the set is convex, and the straight segment between two distributions is the *mixture path* that `02/01` walked for concavity and `02/07`–`02/08` will compare with two other paths. Anatomy, in one breath: **vertices** = certain outcomes ($H = 0$), **edges and faces** = one outcome forbidden (the walls where `02/02`'s $D(p\|q)$ went to infinity), **centre** = the uniform ($H = \log_2 3$). Next we draw the landscapes on it.


## 2. Landscapes on the triangle: entropy, relative entropy, and the uniform law

The triangle is a space, so a function of $p$ is a picture. **Entropy** is a hill: $H(p) = \log_2 3$ at the centre, $0$ at the three corners, concave along every segment (`02/01` §2; exercise 2 there asked for this picture — here it is), and its **level sets** — the curves along which $H$ is constant — are the contour lines of that hill. **Relative entropy** to a fixed $p$ is the bowl of `02/02` §3, asymmetric and infinite at the edges. And the **uniform law on the triangle** is what `02/01` and `02/02` sampled from without saying why: the Dirichlet distribution with all parameters equal to $1$ has constant density on the simplex — a flat rain of points — so `rng.dirichlet(np.ones(3))` is "a random distribution, with no preference". The centre of mass of that rain is the uniform distribution. Let's draw the entropy hill with its level sets, the $D(\cdot\|p)$ bowl, and the rain, side by side.


In [ ]:
def simplex_grid(m=300):
    g = np.linspace(0.0, 1.0, m)
    P1, P2 = np.meshgrid(g, g)
    P3 = 1.0 - P1 - P2
    inside = P3 >= 0
    P = np.stack([P1, P2, P3], axis=-1)
    return P, inside

P, inside = simplex_grid()
XY = P @ V                                                   # every grid point mapped to the plane (barycentric)
with np.errstate(divide="ignore", invalid="ignore"):
    Hgrid = np.where(P > 0, -P * np.log2(P), 0.0).sum(-1)    # entropy, 0 log 0 = 0
    p0 = np.array([0.7, 0.2, 0.1])
    Dgrid = np.where(P > 0, P * np.log2(P / p0), 0.0).sum(-1)   # D(q || p0), finite everywhere (p0 > 0)
Hgrid[~inside] = np.nan; Dgrid[~inside] = np.nan

fig, axes = plt.subplots(1, 3, figsize=(18, 6.2), gridspec_kw={"wspace": 0.45})
for ax in axes:
    draw_triangle(ax)
cs = axes[0].tricontourf(XY[inside][:, 0], XY[inside][:, 1], Hgrid[inside], levels=np.linspace(0, np.log2(3), 21), cmap=viz.CMAP_PLAN)
axes[0].tricontour(XY[inside][:, 0], XY[inside][:, 1], Hgrid[inside], levels=[0.5, 1.0, 1.3, 1.5], colors=[COLORS["text"]], linewidths=0.7)
axes[0].scatter(*to_plane([1/3, 1/3, 1/3]), color=COLORS["highlight"], s=60, zorder=5)
fig.colorbar(cs, ax=axes[0], shrink=0.6, pad=0.04, location="bottom", label="H(p)  (bits)", ticks=[0, 0.5, 1.0, 1.5]); axes[0].set_title("Entropy H(p): a concave hill", pad=26)
cs = axes[1].tricontourf(XY[inside][:, 0], XY[inside][:, 1], Dgrid[inside], levels=np.linspace(0, 3.4, 35), cmap=viz.CMAP_COST)   # max log2(1/0.1) = 3.32 at the delta_3 corner
axes[1].tricontour(XY[inside][:, 0], XY[inside][:, 1], Dgrid[inside], levels=[0.1, 0.25, 0.5, 1.0, 2.0], colors=[COLORS["text"]], linewidths=0.7)
axes[1].scatter(*to_plane(p0), color=COLORS["highlight"], s=60, zorder=5)
fig.colorbar(cs, ax=axes[1], shrink=0.6, pad=0.04, location="bottom", label="D(q || p)  (bits)", ticks=[0, 1, 2, 3]); axes[1].set_title("D(q || p), p = (0.7, 0.2, 0.1): an asymmetric bowl", pad=26)
rain = rng.dirichlet(np.ones(3), size=3000)
axes[2].scatter(*(rain @ V).T, s=4, color=viz.SOURCE_COLOR, alpha=0.6)
axes[2].scatter(*to_plane(rain.mean(0)), color=COLORS["highlight"], s=80, zorder=5, label="centre of mass of the rain")
axes[2].legend(loc="lower center", bbox_to_anchor=(0.5, -0.2), frameon=False); axes[2].set_title("dirichlet(1, 1, 1): a flat rain", pad=26)
plt.show()

print(f"entropy at the centre: {shannon_entropy([1/3, 1/3, 1/3]):.4f} = log2 3 = {np.log2(3):.4f};   at a vertex: {shannon_entropy([1, 0, 0]):.4f};   at the edge midpoint (0.5, 0.5, 0): {shannon_entropy([0.5, 0.5, 0]):.4f}")
print(f"uniform rain: mean of 3000 dirichlet(1,1,1) points = {np.round(rain.mean(0), 3)} (the uniform);  fraction in each of the three corner triangles (p_i > 1/2): {np.round([(rain[:, i] > 0.5).mean() for i in range(3)], 3)}  (each has 1/4 of the area)")

**Read the figure and the output.** Left: the entropy hill, $\log_2 3 = 1.585$ bits at the rose centre, sliding to $0$ at the three corners; the level sets are rounded triangles that shrink toward the centre, and the edge midpoint $(0.5, 0.5, 0)$ sits at exactly $1$ bit — a fair coin on two outcomes, the third forbidden. Concavity is visible: along any straight line the surface never dips. Middle: the bowl $D(q\|p)$ around $p = (0.7, 0.2, 0.1)$, with its single zero at $p$ and its level sets reaching *farther toward* the nearby $\delta_1$ corner than toward the others (the bowl is shallow there: $D(\delta_1\|p) = \log_2(1/0.7) = 0.51$ bits against $3.32$ at $\delta_3$) — relative entropy is not a round bowl, and the same picture with the arguments swapped (`02/02` §3) has walls at the edges. Right: three thousand `dirichlet(1,1,1)` points fall as a flat rain — no clustering, each corner triangle (the region $p_i > \tfrac12$, one quarter of the area; the three of them leave the central inverted quarter) catching close to a quarter of the points — and their centre of mass lands on the uniform. That is what "a random distribution" meant in `02/01`'s Jensen check and `02/02`'s convexity check: uniform over *this* triangle.


## 3. Two bits, one tetrahedron, and the independence surface

`02/03` measured how far a joint table is from the product of its marginals. On the simplex that sentence is a picture. The joint tables of two bits, $p(x, y)$ with four entries, live in $\Delta_3$ — a **tetrahedron** with vertices $\delta_{00}, \delta_{01}, \delta_{10}, \delta_{11}$. Inside it, the *product* tables $p_X \otimes p_Y = (ab,\ a(1-b),\ (1-a)b,\ (1-a)(1-b))$ form a two-parameter family — a curved **surface**, the **independence surface**, cut out by one equation: $p_{00}\,p_{11} = p_{01}\,p_{10}$, the table's $2\times2$ determinant is zero — the classical twin of `01/17`'s amplitude-matrix determinant test (algebraic geometers call it the Segre quadric; further reading in the references). Mutual information is the relative entropy from a table to its own projection onto that surface: it is $0$ on the surface and grows away from it. The correlated coins of `01/16`, $\mathrm{diag}(\tfrac12, 0, 0, \tfrac12)$, sit on the edge joining $\delta_{00}$ to $\delta_{11}$ — as far from the surface as a two-bit table can be ($1$ bit, the ceiling of `02/03` §2).

A tetrahedron is hard to draw on a page, so take a **slice**: fix the two marginals to be fair coins, $p_X = p_Y = (\tfrac12, \tfrac12)$. That leaves one free number, the agreement probability $s = p(0,0) + p(1,1)$, so the slice is a segment from the anti-correlated coins ($s = 0$) through the independent table ($s = \tfrac12$, where the segment *pierces* the independence surface) to the correlated coins ($s = 1$). Let's compute $I$ along it — you know the answer from `02/03`: $1 - h(s)$ — and, for the full tetrahedron, check on random tables that the independence surface is exactly the zero set of $I$ and that the mixture of two product tables is in general *not* a product table (the surface is curved, not flat).


In [ ]:
s_grid = np.linspace(0.0, 1.0, 101)
def slice_table(s):                       # fair marginals, agreement probability s
    return np.array([[s / 2, (1 - s) / 2], [(1 - s) / 2, s / 2]])
I_slice = np.array([mutual_information(slice_table(s)) for s in s_grid])
h = lambda p: shannon_entropy([p, 1 - p])
print(f"slice with fair marginals: I(s) = 1 - h(s) to {np.max(np.abs(I_slice - (1 - np.array([h(s) for s in s_grid])))):.1e};  I = 0 only at s = 1/2 (the independence surface), 1 bit at both ends")

# the independence surface inside the tetrahedron: product tables have I = 0; random tables do not; mixtures of products are not products
prod_tables = [np.outer(rng.dirichlet([1, 1]), rng.dirichlet([1, 1])) for _ in range(500)]
rand_tables = rng.dirichlet(np.ones(4), size=500).reshape(-1, 2, 2)
print(f"500 random PRODUCT tables: max I = {max(mutual_information(T) for T in prod_tables):.1e} bits (on the surface);  max |det| = {max(abs(np.linalg.det(T)) for T in prod_tables):.1e} (the surface's equation)")
print(f"500 random tables of the tetrahedron: median I = {np.median([mutual_information(T) for T in rand_tables]):.4f} bits, max = {max(mutual_information(T) for T in rand_tables):.4f}")
T1, T2 = prod_tables[0], prod_tables[1]
print(f"mixture of two product tables: I = {mutual_information(0.5 * T1 + 0.5 * T2):.4f} bits > 0, det = {np.linalg.det(0.5 * T1 + 0.5 * T2):.4f} -- the surface is curved (a generic chord leaves it; chords between tables sharing a marginal stay on it)")
print(f"the correlated coins diag(1/2, 0, 0, 1/2): I = {mutual_information(slice_table(1.0)):.4f} bit, on the edge delta_00 -- delta_11, the farthest a two-bit table gets from independence")

fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(s_grid, I_slice, color=viz.SOURCE_COLOR, lw=2.4, label="I(X;Y) along the slice")
ax.axvline(0.5, color=COLORS["muted"], ls="--", lw=1); ax.scatter([0.5], [0.0], color=COLORS["highlight"], s=60, zorder=5, label="pierces the independence surface")
ax.set_xlabel("agreement probability s   (0 = anti-correlated coins, 1/2 = independent, 1 = correlated coins)"); ax.set_ylabel("bits")
ax.set_title("A segment through the tetrahedron of two-bit tables, at fair marginals", pad=10); ax.legend(loc="upper center")
plt.show()

**Read the output and the figure.** Along the fair-marginal slice, $I = 1 - h(s)$ to machine precision — `02/03`'s U, now read as *distance from a segment to a surface*: $1$ bit at both ends (the two edges of the tetrahedron where the coins agree always or never), $0$ exactly where the segment pierces the independence surface. Five hundred product tables have $I = 0$ and determinant $0$ to rounding — they *are* the surface; random tables of the tetrahedron sit off it (median $0.07$ bits); and the midpoint of two generic product tables is *not* a product table ($I = 0.009 > 0$, determinant nonzero): the independence surface is curved, so the mixture path between two independent tables generically leaves independence (it stays on the surface only when the two tables share a marginal — the determinant equation is bilinear, so the surface is doubly ruled). Keep this picture for module 04: there, a pair of signals will be embedded as a joint *quantum* state, and the question "how correlated?" is again a distance to a product family — with the quantum mutual information of `02/10` in the role of $I$.


## 4. Four rulers on one triangle — the question of the arc

You already own four ways to say how far apart two points $p, q$ of the triangle are, and you have used each:

- **Euclidean**: $\|p - q\|_2$ — treat the triangle as a flat piece of the plane.
- **Total variation** (`01/14`): $\mathrm{TV} = \tfrac12\|p - q\|_1 = \tfrac12\sum_i|p_i - q_i|$ — the best one-shot guessing advantage.
- **Hellinger** (from `01/14`'s Bhattacharyya overlap $B = \sum_i\sqrt{p_i q_i}$): $\mathrm{Hel} = \sqrt{1 - B}$ in this course's convention (some authors use $\|\sqrt p - \sqrt q\|_2 = \sqrt{2(1 - B)}$, a factor $\sqrt2$ larger) — a distance, triangle inequality included (`02/07` shows why, through a square-root map).
- **Relative entropy** (`02/02`): $D(p\|q)$ — not a distance, asymmetric; we draw the direction $q \mapsto D(p\|q)$, *truth fixed, model varies*, the one that is infinite at every edge.

Draw the **level set** of each around the same centre $p$ — all points at a fixed "distance" from $p$ — and they are four different curves. Near the centre of the triangle, for a small radius, three of them look alike: $\|q - p\|_2^2$, $\mathrm{Hel}^2$ and $D$ are all *quadratic forms* in $q - p$ to leading order, so their level sets are small ellipses, while total variation is polyhedral — its level set is a hexagon at every scale. Near the boundary they part company, because they treat a *rare outcome* differently: Euclidean and TV do not care whether a change of $0.01$ happens to an outcome of probability $0.5$ or $0.01$; Hellinger and KL care a great deal (a relative change of $100\%$ versus $2\%$). That is the question of the arc, made visible: **which ruler is the natural one for distributions?** `02/06` answers with a criterion — distance should be *statistical distinguishability*, unchanged by relabelling or by splitting an outcome into sub-outcomes — and builds the metric that criterion forces; `02/07` walks its shortest paths; and `02/08` shows there is a *second*, entirely different answer once the outcomes have a geometry of their own. Let's draw the four level sets around two centres, one near the middle and one near an edge.


In [ ]:
def level_sets(p):
    # distances from every grid point q to p, on the simplex grid; returns dict name -> field
    with np.errstate(divide="ignore", invalid="ignore"):
        euc = np.linalg.norm(P - p, axis=-1)
        tv = 0.5 * np.abs(P - p).sum(-1)
        hel = np.sqrt(np.clip(1.0 - np.sqrt(np.clip(P, 0, None) * p).sum(-1), 0, None))
        kl = np.minimum((p * np.log2(p / P)).sum(-1), 10.0)           # D(p || q): truth p fixed, model q varies; +inf wherever q has a zero (capped at 10 bits so the contour routine accepts it)
    fields = {"Euclidean": euc, "total variation": tv, "Hellinger": hel, "KL  D(p || q)": kl}
    for k in fields:
        fields[k] = np.where(inside, fields[k], np.nan)
    return fields

centres = {"near the middle: p = (0.4, 0.35, 0.25)": np.array([0.4, 0.35, 0.25]),
           "near an edge (zoomed): p = (0.55, 0.42, 0.03)": np.array([0.55, 0.42, 0.03])}
levels = {"Euclidean": 0.15, "total variation": 0.12, "Hellinger": 0.07, "KL  D(p || q)": 0.03}   # one contour each, chosen so the curves are comparable in size
palette = {"Euclidean": COLORS["muted"], "total variation": viz.TARGET_COLOR, "Hellinger": viz.FLOW_COLOR, "KL  D(p || q)": viz.SOURCE_COLOR}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), gridspec_kw={"wspace": 0.3})
for ax, (title, p) in zip(axes, centres.items()):
    draw_triangle(ax, labels=(ax is axes[0]))      # the zoomed panel shows only the edge
    fields = level_sets(p)
    for name, field in fields.items():
        ax.tricontour(XY[inside][:, 0], XY[inside][:, 1], field[inside], levels=[levels[name]], colors=[palette[name]], linewidths=2.2)
        ax.plot([], [], color=palette[name], lw=2.2, label=f"{name} = {levels[name]}")
    ax.scatter(*to_plane(p), color=COLORS["highlight"], s=60, zorder=5)
    ax.set_title(title, pad=26); ax.legend(loc="upper right", fontsize=9)
axes[1].set_xlim(0.3, 0.8); axes[1].set_ylim(-0.02, 0.24)        # zoom on the edge region; the triangle's bottom edge is the line y = 0
axes[1].text(0.55, -0.012, "edge q3 = 0", ha="center", va="top", color=COLORS["muted"], fontsize=10)
fig.suptitle("Four rulers, one triangle: the level sets agree near the middle and disagree near the boundary", y=1.0)
plt.show()

# how close does each level set of the right panel come to the edge q3 = 0?  (computed, not read off pixels)
p_edge_c = centres["near an edge (zoomed): p = (0.55, 0.42, 0.03)"]
q3_scan = np.linspace(1e-4, 0.03, 3000); q1_scan = np.linspace(0.3, 0.8, 2000)
def closest_q3(dist_fn, level):
    for q3 in q3_scan:
        Q = np.stack([q1_scan, 1 - q1_scan - q3, np.full_like(q1_scan, q3)], axis=1)
        if dist_fn(Q).min() <= level:
            return q3
    return np.nan
print("closest approach of each level set to the edge q3 = 0 (the centre has p3 = 0.03):")
print(f"  Euclidean {levels['Euclidean']} and TV {levels['total variation']}: cross the edge (the minimum Euclidean distance from p to the edge is {np.sqrt(1.5) * p_edge_c[2]:.4f} < 0.15)")
print(f"  Hellinger {levels['Hellinger']}: stops at q3 = {closest_q3(lambda Q: np.sqrt(1 - np.sqrt(p_edge_c * Q).sum(1)), levels['Hellinger']):.4f};   the edge itself is at Hellinger distance {np.sqrt(1 - np.sqrt(1 - p_edge_c[2])):.4f}")
print(f"  KL D(p||q) = {levels['KL  D(p || q)']}: stops at q3 = {closest_q3(lambda Q: (p_edge_c * np.log2(p_edge_c / Q)).sum(1), levels['KL  D(p || q)']):.4f};   the edge itself is at D = inf")

# the same disagreement in numbers: a shift of 0.02 of mass, applied to a common outcome vs a rare one
p = np.array([0.5, 0.48, 0.02])
q_common = np.array([0.48, 0.50, 0.02])   # move 0.02 between the two common outcomes
q_rare = np.array([0.50, 0.50, 0.00])     # move the rare outcome's 0.02 onto a common one
for name, q in [("shift 0.02 between common outcomes", q_common), ("remove the rare outcome's 0.02   ", q_rare)]:
    B = np.sqrt(p * q).sum()
    print(f"{name}:  Euclidean {np.linalg.norm(p - q):.4f}   TV {0.5 * np.abs(p - q).sum():.4f}   Hellinger {np.sqrt(1 - B):.4f}   D(p||q) {kl_divergence(p, q):.4f}")

**Read the figure and the output.** Around the centre-ish point (left) the four level sets are four loops of similar size and shape — the rulers broadly agree on what "nearby" means: three rounded and one hexagonal, as the quadratic-versus-polyhedral remark predicted, and the Hellinger and KL loops lie almost on top of each other (they are the *same* quadratic form near a point — `02/06` names it). Around the point near the bottom edge (right, zoomed) they split: the Euclidean circle and the TV hexagon run straight into the edge and are cut off by it, as if the boundary were an arbitrary line drawn across a flat plane; the Hellinger and KL curves flatten against the edge and stop short of it — by margins the figure can barely resolve, so they are *computed* in the output: the Hellinger $0.07$ contour stops at $q_3 = 0.0056$ and the KL $0.03$ contour at $q_3 = 0.0070$, while the Euclidean and TV contours cross $q_3 = 0$. The two statistical rulers differ in *how* far the edge is: in this convention the edge $q_3 = 0$ sits at Hellinger distance $\sqrt{1 - \sqrt{0.97}} = 0.123$ from $p$ — more than three times the Euclidean $0.037$, *farther* but reachable — whereas for the truth $p$ every model $q$ with $q_3 = 0$ costs infinitely many bits, $D(p\|q) = \infty$: the KL level sets squeeze toward the edge for ever without reaching it. The numbers say the same thing with one move of $0.02$ of mass: shifted between two common outcomes it costs Euclidean $0.028$, TV $0.02$, Hellinger $0.014$, $D = 0.0012$ bits; taken away from the rare outcome — the *same* $0.02$ of mass — Euclidean and TV do not change at all, Hellinger is seven times larger, and $D(p\|q)$ is **infinite**. Flat rulers count mass; statistical rulers count *surprise* — and they disagree with each other about *how* far the boundary is (Hellinger: finite but stretched; KL: infinite in this direction). The arc now has its question and its suspects, and `02/06` brings the criterion that picks the answer.


## Your turn

1. **Your own points, and the entropy hill by hand.** Add $(0.5, 0.25, 0.25)$ and $(0.1, 0.1, 0.8)$ to the examples and plot them with `viz.plot_simplex_points`; then, without the helper, compute the plane coordinates by the barycentric map and check. Walk the straight line from $(0.5, 0.25, 0.25)$ to the uniform in $50$ steps and plot $H$ along it: is it concave, and where is the maximum? Then walk from a vertex to the opposite edge's midpoint — same question.
2. **Higher simplices.** $\Delta_3$ (four outcomes) is a tetrahedron: how many vertices, edges, faces? In general, how many $k$-dimensional faces does $\Delta_{n-1}$ have (hint: choose which outcomes are *allowed*)? Check your formula by counting the faces of $\Delta_3$ on which the entropy is at most $1$ bit everywhere.
3. **The rain is not always flat.** Sample `rng.dirichlet([1, 1, 1])`, `rng.dirichlet([5, 5, 5])` and `rng.dirichlet([0.3, 0.3, 0.3])`, $3000$ points each, and plot the three rains on the triangle side by side. Describe where each concentrates and compute the mean entropy of each rain. Which of the three would be a fair "random distribution" for a Jensen check, and why?
4. **Harder: the independence surface in 3-D, and the quantum slice.** (a) Parametrise the independence surface of two bits by $(a, b)$, map it into the tetrahedron with a $4$-vertex barycentric map into $\mathbb R^3$ (vertices of a regular tetrahedron), and draw it with `matplotlib`'s 3-D axes together with the segment of section 3 — where does the segment pierce it? (b) The quantum side: the qubit state space of `01/13` is the Bloch *ball*, and the classical two-outcome simplex is the segment of diagonal states along the $z$-axis. Using `qot_course.quantum.density`, draw $100$ random qubit density matrices (random Bloch vectors inside the ball) and mark where their *diagonals* $(\rho_{00}, \rho_{11})$ fall on that segment; explain in one sentence why `01/13`'s $|+\rangle$ and $I/2$ land on the same point of the segment, and what this says about what the simplex cannot see.


## What you built

- You defined the probability simplex, counted its dimension, proved it is convex by hand (mixtures stay inside), and placed three-outcome distributions on the triangle by the barycentric map — a distribution is its own set of weights — checking the helper against your own coordinates.
- You read its anatomy and drew the landscapes of the last four notebooks on it: the entropy hill (peak $\log_2 3$ at the centre, $0$ at the corners), the asymmetric KL bowl, and the flat rain of `dirichlet(1,1,1)` whose centre of mass is the uniform.
- You placed the two-bit tables of `01/16` in their tetrahedron, saw the product tables form a curved independence surface, and re-read mutual information as the distance to it — $1 - h(s)$ along the fair-marginal segment.
- You drew four candidate distances around one point and watched them agree near the middle and disagree near the boundary — flat rulers count mass, statistical rulers count surprise — and posed the question the arc answers.
- You can place the simplex inside its quantum counterpart: the density matrices of a $d$-level system, $\rho \ge 0$ with $\mathrm{tr}\rho = 1$, form a convex body of dimension $d^2 - 1$.
- In any basis, the density matrices that are diagonal in that basis form a slice of the quantum state space, and that slice is the simplex: a $\rho$ diagonal in a shared basis *is* a $p$ (`01/12`).
- In the Bloch ball of `01/13`, pure states sit on the surface, mixtures fill the inside, and $I/2$ sits at the centre.
- On that diagonal slice the quantum distances come back to classical ones: the trace distance of `01/14` becomes the total variation exactly.
- On the same slice, the Bures distance of `02/12` becomes $\sqrt2$ times this notebook's Hellinger distance: with the course's conventions, $d_B = \sqrt{2(1 - B)} = \sqrt2\,\mathrm{Hel}$.
- For $n$ outcomes the same hill peaks at $\log_2 n$ at the centre, the ceiling you met in `02/01`.

The geometry arc has its stage: a convex triangle, convex in the sense `02/01`–`02/02` already used, on which four rulers disagree exactly where probability is small. Which one is *natural* is the next notebook's business.


**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| probability simplex $\Delta_{n-1} = \{p \in \mathbb R^n : p_i \ge 0, \sum_i p_i = 1\}$ | state space of a $d$-level system: density matrices $\rho \ge 0$, $\mathrm{tr}\rho = 1$ |

## What's next

Euclidean distance treats the triangle as flat, and the right-hand panel above says it is not the natural choice: the rulers that count surprise flatten against the edges. In `02/06_fisher_rao_metric` we ask for the ruler that measures *statistical distinguishability* — how well samples tell two nearby distributions apart — and find the Fisher information metric: its coefficients blow up at the boundary, every step near a corner is expensive, and the triangle is genuinely curved (the corners themselves stay at a finite distance, as `02/07`'s closed form will show).


## References

- S.-i. Amari, *Information Geometry and Its Applications*, Applied Mathematical Sciences 194, Springer (2016) — ch. 1–2: the simplex as a manifold, mixture and exponential coordinates. doi:10.1007/978-4-431-55978-8
- F. Nielsen, *An elementary introduction to information geometry*, Entropy **22**, 1100 (2020) — the simplex, divergences and the Fisher–Rao metric in one survey. doi:10.3390/e22101100
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §2.7 (concavity of $H$, convexity of $D$ as statements on the simplex), §11.6 (Pinsker and the relation between distances). doi:10.1002/047174882X
- D. Geiger, C. Meek & B. Sturmfels, *On the toric algebra of graphical models*, Ann. Statist. **34**, 1463–1492 (2006) — the independence model as a (Segre) variety inside the simplex. doi:10.1214/009053606000000263
- I. Bengtsson & K. Życzkowski, *Geometry of Quantum States*, 2nd ed., Cambridge University Press (2017) — ch. 1–2 (the simplex and its geometries), ch. 8 (the space of density matrices, of which the simplex is a slice). doi:10.1017/9781139207010

**Previous:** `notebooks/02_InformationAndGeometry/04_transfer_entropy.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/06_fisher_rao_metric.ipynb`
